# Week 4: Tools & MCP — Building the Agent's Hands

## Phase 2, Week 4 of the AI Agents from Scratch Learning Path

**Goal**: Master tool design, build production-quality tools, understand MCP (Model Context Protocol), and create a full MCP server with 5+ tools.

### What You'll Learn

| Section | Topic | What You'll Build |
|---------|-------|-------------------|
| 1 | Tool Design Principles | Mental model for great agent tools |
| 2 | Production-Quality Tools | Type-safe tools with Pydantic, validation, error handling |
| 3 | Auto-Schema Generation | Decorator that auto-generates OpenAI tool schemas from type hints |
| 4 | Tool Middleware & Composition | Logging, rate limiting, auth, and tool chaining |
| 5 | MCP Protocol Deep Dive | Understanding the universal tool standard |
| 6 | Building an MCP Server | Full MCP server with 5+ tools from scratch |
| 7 | Building an MCP Client | Connect, discover, and invoke tools over MCP |
| 8 | Capstone: MCP-Powered Agent | Complete agent using MCP for tool discovery and execution |

### Prerequisites
- Week 3 completed (agent loops, ReAct, tool registry basics)
- Python 3.10+
- `pip install pydantic httpx aiohttp`
- For MCP sections: `pip install mcp` (optional — we simulate first)

In [2]:
# ============================================================
# Week 4 — Setup & Imports
# ============================================================
import json
import time
import asyncio
import hashlib
import inspect
import logging
import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Optional, get_type_hints
from functools import wraps

# We'll use Pydantic heavily this week for tool input validation
from pydantic import BaseModel, Field, field_validator

print("✅ Week 4 imports loaded — ready to build tools!")

✅ Week 4 imports loaded — ready to build tools!


---
# Section 1: Tool Design Principles

## What Makes a Great Agent Tool?

An agent tool is a function the LLM can call. But not all functions make good tools.

### The CLEAR Framework for Tool Design

```
C — Composable       → Tools should do ONE thing well, combinable with others
L — Labeled clearly  → Name and description must be unambiguous to the LLM
E — Error-safe       → Never crash — always return structured errors
A — Atomic           → Each call is self-contained, no hidden state
R — Reversible-aware → Side effects are documented and ideally reversible
```

### What the LLM Sees vs What Actually Runs

```
┌──────────────────────────────────────────────────────────────────┐
│  What the LLM "sees" (JSON Schema)                               │
│                                                                  │
│  {                                                               │
│    "name": "search_products",                                    │
│    "description": "Search the product catalog by query",         │
│    "parameters": {                                               │
│      "query": {"type": "string", "description": "Search terms"}, │
│      "limit": {"type": "integer", "default": 10}                 │
│    }                                                             │
│  }                                                               │
│                                                                  │
│  The LLM ONLY has this schema to decide:                         │
│    1. WHETHER to call this tool                                  │
│    2. WHAT arguments to pass                                     │
│                                                                  │
│  It never sees the implementation code!                           │
└──────────────────────────────────────────────────────────────────┘
         │
         ▼
┌──────────────────────────────────────────────────────────────────┐
│  What actually runs (Python function)                            │
│                                                                  │
│  def search_products(query: str, limit: int = 10) -> str:        │
│      validated = ProductQuery(query=query, limit=limit)          │
│      results = db.search(validated.query, validated.limit)       │
│      return json.dumps(results)                                  │
│                                                                  │
│  This can do ANYTHING: DB queries, API calls, file I/O, etc.    │
└──────────────────────────────────────────────────────────────────┘
```

### Common Tool Design Mistakes

| Mistake | Why It's Bad | Fix |
|---------|-------------|-----|
| Vague name: `do_stuff()` | LLM can't decide when to use it | `search_knowledge_base()` |
| No description | LLM has to guess from name alone | Always add detailed description |
| Too many params | LLM fills them wrong | Max 5-6 params, use defaults |
| Giant return values | Wastes context window tokens | Summarize, paginate, limit |
| No error handling | Agent loop crashes | Always return error as string |
| Side effects undocumented | Agent triggers unintended actions | Document in description |
| Boolean params | LLMs struggle with true/false | Use enum strings instead |

In [3]:
# ============================================================
# Section 1: Tool Design — Good vs Bad Tools (Hands-on Demo)
# ============================================================

# ❌ BAD TOOL DESIGN — Let's see why each is problematic

def bad_tool_1(x):
    """No type hints, vague name, no description concept."""
    return str(x * 2)

def bad_tool_2(query: str, include_metadata: bool = True, 
               sort_by: str = "relevance", max_results: int = 100,
               include_related: bool = True, filter_category: str = "",
               date_range: str = "", language: str = "en",
               include_snippets: bool = True) -> str:
    """Too many parameters — LLM will fill many of these wrong."""
    return "too complex"

def bad_tool_3(city: str) -> dict:
    """Returns raw dict — if the dict is huge, wastes context tokens.
    Also returns dict, not string — some frameworks expect string."""
    return {
        "city": city,
        "weather": "sunny",
        "temp": 72,
        "history": [{"day": i, "temp": 70+i} for i in range(365)]  # 365 entries!
    }

# ✅ GOOD TOOL DESIGN — Following the CLEAR framework

def get_current_weather(city: str, units: str = "celsius") -> str:
    """Get the current weather for a specific city.
    
    Use this tool when the user asks about current weather conditions, 
    temperature, or forecasts for a specific location.
    
    Args:
        city: The city name (e.g., "Paris", "Tokyo", "New York")
        units: Temperature units — either "celsius" or "fahrenheit"
    
    Returns:
        JSON string with temperature, conditions, and humidity
    """
    # Simulated weather data
    weather_db = {
        "paris": {"temp_c": 22, "conditions": "Sunny", "humidity": 45},
        "tokyo": {"temp_c": 28, "conditions": "Humid", "humidity": 78},
        "london": {"temp_c": 15, "conditions": "Rainy", "humidity": 82},
        "new york": {"temp_c": 25, "conditions": "Partly Cloudy", "humidity": 55},
    }
    
    data = weather_db.get(city.lower())
    if not data:
        return json.dumps({"error": f"Weather data not available for '{city}'. Try: Paris, Tokyo, London, New York"})
    
    temp = data["temp_c"]
    if units == "fahrenheit":
        temp = round(temp * 9/5 + 32)
    
    return json.dumps({
        "city": city,
        "temperature": temp,
        "units": units,
        "conditions": data["conditions"],
        "humidity": f"{data['humidity']}%"
    })

# Let's compare
print("=== BAD TOOL: Returns huge uncontrolled output ===")
bad_result = bad_tool_3("Paris")
print(f"Output size: {len(str(bad_result))} chars")
print(f"Contains {len(bad_result['history'])} history entries (wastes tokens!)")

print("\n=== GOOD TOOL: Returns focused, structured output ===")
good_result = get_current_weather("Paris")
print(f"Output size: {len(good_result)} chars")
print(f"Output: {good_result}")

print("\n=== GOOD TOOL: Handles errors gracefully ===")
error_result = get_current_weather("Atlantis")
print(f"Error output: {error_result}")

=== BAD TOOL: Returns huge uncontrolled output ===
Output size: 9777 chars
Contains 365 history entries (wastes tokens!)

=== GOOD TOOL: Returns focused, structured output ===
Output size: 98 chars
Output: {"city": "Paris", "temperature": 22, "units": "celsius", "conditions": "Sunny", "humidity": "45%"}

=== GOOD TOOL: Handles errors gracefully ===
Error output: {"error": "Weather data not available for 'Atlantis'. Try: Paris, Tokyo, London, New York"}


---
# Section 2: Building Production-Quality Tools with Pydantic

## Why Pydantic for Tools?

LLMs generate arguments as JSON. But JSON has no validation — the LLM might send:
- Wrong types (`"limit": "five"` instead of `"limit": 5`)
- Missing required fields
- Values out of range (`"limit": -1` or `"limit": 999999`)
- Malicious inputs (SQL injection, path traversal)

**Pydantic** validates and coerces inputs automatically, giving us type-safe tools.

```
LLM generates JSON args  →  Pydantic validates  →  Tool executes safely
     {"query": "...",         ✅ Types checked        No crashes,
      "limit": "5"}          ✅ Coerced to int        no injection,
                              ✅ Range checked         clean results
```

In [4]:
# ============================================================
# Section 2: Pydantic-Validated Tools
# ============================================================

# --- 2.1: Define Input Models with Pydantic ---

class SearchProductsInput(BaseModel):
    """Input schema for the search_products tool."""
    query: str = Field(
        ...,  # required
        min_length=1,
        max_length=200,
        description="Search terms to find products"
    )
    category: Optional[str] = Field(
        default=None,
        description="Product category filter: electronics, books, clothing"
    )
    max_results: int = Field(
        default=10,
        ge=1,       # >= 1
        le=50,      # <= 50
        description="Maximum number of results (1-50)"
    )
    sort_by: str = Field(
        default="relevance",
        pattern=r"^(relevance|price_asc|price_desc|rating)$",
        description="Sort order: relevance, price_asc, price_desc, or rating"
    )

    @field_validator("query")
    @classmethod
    def sanitize_query(cls, v: str) -> str:
        """Prevent injection attacks in search queries."""
        # Remove potential SQL/NoSQL injection patterns
        dangerous = [";", "--", "'", '"', "DROP", "DELETE", "$where", "$gt"]
        cleaned = v
        for pattern in dangerous:
            cleaned = cleaned.replace(pattern, "")
        return cleaned.strip()


class CalculateInput(BaseModel):
    """Input schema for the calculate tool."""
    expression: str = Field(
        ...,
        max_length=500,
        description="Mathematical expression to evaluate (e.g., '2 + 3 * 4')"
    )

    @field_validator("expression")
    @classmethod
    def validate_expression(cls, v: str) -> str:
        """Only allow safe mathematical expressions."""
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in v):
            raise ValueError(f"Expression contains invalid characters. Only numbers and +-*/.() are allowed.")
        return v


class SendEmailInput(BaseModel):
    """Input schema for the send_email tool — a side-effect tool."""
    to: str = Field(..., description="Recipient email address")
    subject: str = Field(..., min_length=1, max_length=200, description="Email subject")
    body: str = Field(..., min_length=1, max_length=5000, description="Email body text")
    
    @field_validator("to")
    @classmethod
    def validate_email(cls, v: str) -> str:
        pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        if not re.match(pattern, v):
            raise ValueError(f"Invalid email address: {v}")
        return v.lower()


# --- 2.2: Build tools using these input models ---

# Simulated product database
PRODUCT_DB = [
    {"id": 1, "name": "Wireless Headphones", "category": "electronics", "price": 79.99, "rating": 4.5},
    {"id": 2, "name": "Python Crash Course", "category": "books", "price": 29.99, "rating": 4.8},
    {"id": 3, "name": "USB-C Hub", "category": "electronics", "price": 34.99, "rating": 4.2},
    {"id": 4, "name": "Mechanical Keyboard", "category": "electronics", "price": 129.99, "rating": 4.7},
    {"id": 5, "name": "AI Engineering Book", "category": "books", "price": 49.99, "rating": 4.9},
    {"id": 6, "name": "Cotton T-Shirt", "category": "clothing", "price": 19.99, "rating": 4.0},
    {"id": 7, "name": "Smart Watch", "category": "electronics", "price": 199.99, "rating": 4.6},
    {"id": 8, "name": "Hiking Boots", "category": "clothing", "price": 89.99, "rating": 4.3},
]


def search_products(query: str, category: str = None, 
                     max_results: int = 10, sort_by: str = "relevance") -> str:
    """Search the product catalog. Pydantic validates inputs BEFORE this runs."""
    # Validate inputs through Pydantic
    try:
        validated = SearchProductsInput(
            query=query, category=category,
            max_results=max_results, sort_by=sort_by
        )
    except Exception as e:
        return json.dumps({"error": f"Invalid input: {e}"})
    
    # Filter by query
    results = [
        p for p in PRODUCT_DB 
        if validated.query.lower() in p["name"].lower()
    ]
    
    # Filter by category
    if validated.category:
        results = [p for p in results if p["category"] == validated.category]
    
    # Sort
    sort_keys = {
        "relevance": lambda x: x["name"].lower().find(validated.query.lower()),
        "price_asc": lambda x: x["price"],
        "price_desc": lambda x: -x["price"],
        "rating": lambda x: -x["rating"],
    }
    results.sort(key=sort_keys.get(validated.sort_by, sort_keys["relevance"]))
    
    # Limit
    results = results[:validated.max_results]
    
    return json.dumps({
        "query": validated.query,
        "count": len(results),
        "products": results
    })


def calculate(expression: str) -> str:
    """Safely evaluate a math expression."""
    try:
        validated = CalculateInput(expression=expression)
        # Use eval only on validated safe expressions
        result = eval(validated.expression, {"__builtins__": {}}, {})
        return json.dumps({"expression": expression, "result": result})
    except ValueError as e:
        return json.dumps({"error": str(e)})
    except Exception as e:
        return json.dumps({"error": f"Calculation failed: {e}"})


# --- 2.3: Test the tools ---

print("=== Successful tool calls ===")
print(search_products("book", sort_by="rating", max_results=3))
print()
print(calculate("(2 + 3) * 4 / 2"))

print("\n=== Pydantic catches bad inputs ===")
# SQL injection attempt
print(search_products("'; DROP TABLE products; --"))

# Invalid expression
print(calculate("import os; os.system('rm -rf /')"))

# Out of range
print(search_products("test", max_results=1000))

print("\n=== Email validation ===")
try:
    validated = SendEmailInput(to="not-an-email", subject="Hi", body="Test")
except Exception as e:
    print(f"Caught: {e}")

=== Successful tool calls ===
{"query": "book", "count": 1, "products": [{"id": 5, "name": "AI Engineering Book", "category": "books", "price": 49.99, "rating": 4.9}]}

{"expression": "(2 + 3) * 4 / 2", "result": 10.0}

=== Pydantic catches bad inputs ===
{"query": "TABLE products", "count": 0, "products": []}
{"error": "1 validation error for CalculateInput\nexpression\n  Value error, Expression contains invalid characters. Only numbers and +-*/.() are allowed. [type=value_error, input_value=\"import os; os.system('rm -rf /')\", input_type=str]\n    For further information visit https://errors.pydantic.dev/2.12/v/value_error"}
{"error": "Invalid input: 1 validation error for SearchProductsInput\nmax_results\n  Input should be less than or equal to 50 [type=less_than_equal, input_value=1000, input_type=int]\n    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal"}

=== Email validation ===
Caught: 1 validation error for SendEmailInput
to
  Value error, Inv

---
# Section 3: Auto-Generating Tool Schemas from Python Functions

## The Schema Problem

Every tool needs TWO things:
1. **The Python function** (what runs)
2. **The JSON schema** (what the LLM sees)

Writing these separately is error-prone — they get out of sync.

**Solution**: Build a `@tool` decorator that inspects type hints and docstrings to auto-generate schemas.

```
@tool
def search_products(query: str, limit: int = 10) -> str:
    """Search products by keyword."""
    ...

# Decorator auto-generates:
# {
#   "type": "function",
#   "function": {
#     "name": "search_products",
#     "description": "Search products by keyword.",
#     "parameters": {
#       "type": "object",
#       "properties": {
#         "query": {"type": "string", "description": "query parameter"},
#         "limit": {"type": "integer", "description": "limit parameter", "default": 10}
#       },
#       "required": ["query"]
#     }
#   }
# }
```

In [5]:
# ============================================================
# Section 3: The @tool Decorator — Auto Schema Generation
# ============================================================

# Python type → JSON Schema type mapping
PYTHON_TO_JSON_TYPE = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}


def parse_docstring_params(docstring: str) -> dict[str, str]:
    """Extract parameter descriptions from Google-style docstrings.
    
    Parses:
        Args:
            query: Search terms to find products
            limit: Maximum number of results
    """
    param_descriptions = {}
    if not docstring:
        return param_descriptions
    
    in_args = False
    for line in docstring.split("\n"):
        stripped = line.strip()
        if stripped.lower().startswith("args:"):
            in_args = True
            continue
        if in_args:
            if stripped.lower().startswith("returns:") or stripped.lower().startswith("raises:"):
                break
            # Match "param_name: description" or "param_name (type): description"
            match = re.match(r"(\w+)(?:\s*\([^)]*\))?\s*:\s*(.+)", stripped)
            if match:
                param_descriptions[match.group(1)] = match.group(2).strip()
    
    return param_descriptions


def tool(func: Callable = None, *, name: str = None, description: str = None):
    """Decorator that auto-generates an OpenAI-compatible tool schema.
    
    Usage:
        @tool
        def my_func(param: str) -> str:
            '''Description here.'''
            ...
        
        # Access the schema
        my_func.schema  # → OpenAI tool schema dict
    """
    def decorator(fn: Callable) -> Callable:
        # Get function signature
        sig = inspect.signature(fn)
        hints = get_type_hints(fn)
        
        # Parse docstring for parameter descriptions
        doc = inspect.getdoc(fn) or ""
        param_docs = parse_docstring_params(doc)
        
        # Extract the first line of docstring as description
        func_desc = description or doc.split("\n")[0] if doc else f"{fn.__name__} tool"
        
        # Build parameters schema
        properties = {}
        required = []
        
        for param_name, param in sig.parameters.items():
            if param_name == "self":
                continue
            
            # Get type
            param_type = hints.get(param_name, str)
            json_type = PYTHON_TO_JSON_TYPE.get(param_type, "string")
            
            # Build property
            prop = {
                "type": json_type,
                "description": param_docs.get(param_name, f"{param_name} parameter")
            }
            
            # Check for default
            if param.default is inspect.Parameter.empty:
                required.append(param_name)
            else:
                prop["default"] = param.default
            
            properties[param_name] = prop
        
        # Build the schema
        schema = {
            "type": "function",
            "function": {
                "name": name or fn.__name__,
                "description": func_desc,
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": required
                }
            }
        }
        
        # Attach schema to the function
        @wraps(fn)
        def wrapper(*args, **kwargs):
            return fn(*args, **kwargs)
        
        wrapper.schema = schema
        wrapper.tool_name = name or fn.__name__
        wrapper.original = fn
        return wrapper
    
    # Handle both @tool and @tool(name="...", description="...")
    if func is not None:
        return decorator(func)
    return decorator


# --- Let's use it! ---

@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price for a given ticker symbol.
    
    Args:
        symbol: Stock ticker symbol (e.g., AAPL, GOOGL, MSFT)
    
    Returns:
        JSON with current price and change
    """
    prices = {
        "AAPL": {"price": 178.50, "change": "+1.2%"},
        "GOOGL": {"price": 141.80, "change": "-0.5%"},
        "MSFT": {"price": 378.90, "change": "+0.8%"},
    }
    data = prices.get(symbol.upper())
    if not data:
        return json.dumps({"error": f"Unknown symbol: {symbol}"})
    return json.dumps({"symbol": symbol.upper(), **data})


@tool
def search_knowledge_base(query: str, max_results: int = 5) -> str:
    """Search the internal knowledge base for relevant documents.
    
    Args:
        query: Natural language search query
        max_results: Maximum documents to return (default 5)
    
    Returns:
        JSON array of matching documents with snippets
    """
    docs = [
        {"title": "Getting Started Guide", "snippet": "Welcome to our platform..."},
        {"title": "API Reference", "snippet": "REST API endpoints for..."},
        {"title": "Troubleshooting FAQ", "snippet": "Common issues and solutions..."},
    ]
    return json.dumps({"results": docs[:max_results], "total": len(docs)})


@tool(name="calculate_math", description="Evaluate a mathematical expression safely")
def safe_calculate(expression: str) -> str:
    """Calculate the result of a math expression.
    
    Args:
        expression: Mathematical expression using numbers and +-*/()
    """
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return json.dumps({"error": "Invalid characters in expression"})
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})


@tool
def create_reminder(title: str, due_date: str, priority: str = "medium") -> str:
    """Create a new reminder for the user.
    
    Args:
        title: What to be reminded about
        due_date: When the reminder is due (YYYY-MM-DD format)
        priority: Priority level: low, medium, or high
    """
    return json.dumps({
        "status": "created",
        "reminder": {"title": title, "due_date": due_date, "priority": priority}
    })


@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's profile information.
    
    Args:
        user_id: The unique user identifier
    """
    profiles = {
        "user_001": {"name": "Alice", "email": "alice@example.com", "plan": "pro"},
        "user_002": {"name": "Bob", "email": "bob@example.com", "plan": "free"},
    }
    return json.dumps(profiles.get(user_id, {"error": f"User {user_id} not found"}))


# --- Show auto-generated schemas ---

print("=" * 60)
print("AUTO-GENERATED TOOL SCHEMAS")
print("=" * 60)

for t in [get_stock_price, search_knowledge_base, safe_calculate, create_reminder, get_user_profile]:
    schema = t.schema
    print(f"\n📦 {schema['function']['name']}")
    print(f"   Description: {schema['function']['description']}")
    params = schema['function']['parameters']
    print(f"   Parameters:")
    for pname, pinfo in params['properties'].items():
        req = "REQUIRED" if pname in params.get('required', []) else f"default={pinfo.get('default', '?')}"
        print(f"     - {pname}: {pinfo['type']} ({req}) — {pinfo['description']}")

print("\n\n--- Full schema for get_stock_price (what gets sent to OpenAI) ---")
print(json.dumps(get_stock_price.schema, indent=2))

AUTO-GENERATED TOOL SCHEMAS

📦 get_stock_price
   Description: Get the current stock price for a given ticker symbol.
   Parameters:
     - symbol: string (REQUIRED) — Stock ticker symbol (e.g., AAPL, GOOGL, MSFT)

📦 search_knowledge_base
   Description: Search the internal knowledge base for relevant documents.
   Parameters:
     - query: string (REQUIRED) — Natural language search query
     - max_results: integer (default=5) — Maximum documents to return (default 5)

📦 calculate_math
   Description: Evaluate a mathematical expression safely
   Parameters:
     - expression: string (REQUIRED) — Mathematical expression using numbers and +-*/()

📦 create_reminder
   Description: Create a new reminder for the user.
   Parameters:
     - title: string (REQUIRED) — What to be reminded about
     - due_date: string (REQUIRED) — When the reminder is due (YYYY-MM-DD format)
     - priority: string (default=medium) — Priority level: low, medium, or high

📦 get_user_profile
   Description: Re

### 3.2: Pydantic-Based Schema Generation (Advanced)

The `@tool` decorator uses type hints. But for complex tools, we can use **Pydantic models** to generate even richer schemas with nested objects, enums, and constraints.

In [6]:
# ============================================================
# Section 3.2: Pydantic Model → OpenAI Tool Schema
# ============================================================

from pydantic import BaseModel, Field
from enum import Enum
from typing import Optional


class Priority(str, Enum):
    """Task priority levels."""
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"


class TaskFilter(BaseModel):
    """Filter criteria for searching tasks."""
    status: Optional[str] = Field(None, description="Filter by status: open, in_progress, done")
    assignee: Optional[str] = Field(None, description="Filter by assignee name")


class CreateTaskInput(BaseModel):
    """Create a new task in the project management system."""
    title: str = Field(..., min_length=1, max_length=200, description="Task title")
    description: str = Field(default="", max_length=2000, description="Detailed task description")
    priority: Priority = Field(default=Priority.MEDIUM, description="Task priority level")
    assignee: Optional[str] = Field(None, description="Person to assign the task to")
    tags: list[str] = Field(default_factory=list, description="Tags for categorization")


def pydantic_to_openai_schema(model: type[BaseModel], func_name: str, 
                                func_description: str) -> dict:
    """Convert a Pydantic model into an OpenAI function calling schema.
    
    This is how frameworks like LangChain and OpenAI Agents SDK
    generate tool schemas under the hood.
    """
    # Get Pydantic's JSON schema
    json_schema = model.model_json_schema()
    
    # Extract properties and required fields
    properties = {}
    for prop_name, prop_info in json_schema.get("properties", {}).items():
        prop_entry = {}
        
        # Handle type
        if "anyOf" in prop_info:
            # Optional types: anyOf: [{type: X}, {type: null}]
            types = [t for t in prop_info["anyOf"] if t.get("type") != "null"]
            if types:
                prop_entry["type"] = types[0].get("type", "string")
        elif "allOf" in prop_info:
            # Enum reference
            ref = prop_info["allOf"][0].get("$ref", "")
            if ref:
                enum_name = ref.split("/")[-1]
                enum_def = json_schema.get("$defs", {}).get(enum_name, {})
                prop_entry["type"] = "string"
                prop_entry["enum"] = enum_def.get("enum", [])
        else:
            prop_entry["type"] = prop_info.get("type", "string")
        
        # Handle description
        if "description" in prop_info:
            prop_entry["description"] = prop_info["description"]
        
        # Handle default
        if "default" in prop_info:
            prop_entry["default"] = prop_info["default"]
        
        # Handle array items
        if prop_entry.get("type") == "array" and "items" in prop_info:
            prop_entry["items"] = prop_info["items"]
        
        properties[prop_name] = prop_entry
    
    return {
        "type": "function",
        "function": {
            "name": func_name,
            "description": func_description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": json_schema.get("required", [])
            }
        }
    }


# --- Generate schemas from Pydantic models ---

create_task_schema = pydantic_to_openai_schema(
    CreateTaskInput,
    "create_task",
    "Create a new task in the project management system. Use when the user wants to add a todo, task, or work item."
)

print("=== Pydantic → OpenAI Schema ===")
print(json.dumps(create_task_schema, indent=2))

print("\n\n=== Pydantic also validates at runtime ===")

# Valid input
task = CreateTaskInput(
    title="Implement MCP server",
    priority=Priority.HIGH,
    assignee="Alice",
    tags=["backend", "mcp"]
)
print(f"✅ Valid: {task.model_dump()}")

# Invalid input
try:
    bad_task = CreateTaskInput(title="")  # too short
except Exception as e:
    print(f"❌ Caught: {e}")

try:
    bad_task = CreateTaskInput(title="Test", priority="super_urgent")  # invalid enum
except Exception as e:
    print(f"❌ Caught: {e}")

=== Pydantic → OpenAI Schema ===
{
  "type": "function",
  "function": {
    "name": "create_task",
    "description": "Create a new task in the project management system. Use when the user wants to add a todo, task, or work item.",
    "parameters": {
      "type": "object",
      "properties": {
        "title": {
          "type": "string",
          "description": "Task title"
        },
        "description": {
          "type": "string",
          "description": "Detailed task description",
          "default": ""
        },
        "priority": {
          "type": "string",
          "description": "Task priority level",
          "default": "medium"
        },
        "assignee": {
          "type": "string",
          "description": "Person to assign the task to",
          "default": null
        },
        "tags": {
          "type": "array",
          "description": "Tags for categorization",
          "items": {
            "type": "string"
          }
        }
      },
  

---
# Section 4: Advanced Tool Registry with Middleware

## Beyond Simple Registration

In Week 3, we built a basic tool registry. Now we add **middleware** — layers that wrap every tool call:

```
Agent calls tool
  │
  ▼
┌─────────────┐
│  Logging     │  ← Record what was called, when, with what args
├─────────────┤
│  Auth Check  │  ← Does the agent have permission?
├─────────────┤
│  Rate Limit  │  ← Don't exceed API quotas
├─────────────┤
│  Validation  │  ← Pydantic validates inputs
├─────────────┤
│  Execution   │  ← Actually run the tool function
├─────────────┤
│  Timing      │  ← How long did it take?
├─────────────┤
│  Output Trim │  ← Truncate if output too large
└─────────────┘
  │
  ▼
Result returned to agent
```

This is exactly how production agent frameworks handle tool execution.

In [7]:
# ============================================================
# Section 4: Production Tool Registry with Middleware
# ============================================================

@dataclass
class ToolCallRecord:
    """Record of a single tool invocation."""
    tool_name: str
    arguments: dict
    result: str
    duration_ms: float
    timestamp: str
    success: bool
    error: Optional[str] = None


class ToolMiddleware(ABC):
    """Base class for tool middleware. Each middleware wraps tool execution."""
    
    @abstractmethod
    def before(self, tool_name: str, arguments: dict) -> Optional[str]:
        """Called before tool execution. Return error string to block execution."""
        pass
    
    @abstractmethod
    def after(self, tool_name: str, arguments: dict, result: str, duration_ms: float) -> str:
        """Called after tool execution. Can modify the result."""
        pass


class LoggingMiddleware(ToolMiddleware):
    """Logs every tool call for debugging and auditing."""
    
    def __init__(self):
        self.logs: list[ToolCallRecord] = []
    
    def before(self, tool_name: str, arguments: dict) -> Optional[str]:
        print(f"  🔧 Calling: {tool_name}({json.dumps(arguments)[:100]}...)")
        return None
    
    def after(self, tool_name: str, arguments: dict, result: str, duration_ms: float) -> str:
        record = ToolCallRecord(
            tool_name=tool_name,
            arguments=arguments,
            result=result[:200],
            duration_ms=duration_ms,
            timestamp=datetime.now(timezone.utc).isoformat(),
            success='"error"' not in result.lower()
        )
        self.logs.append(record)
        print(f"  ✅ {tool_name} completed in {duration_ms:.1f}ms")
        return result
    
    def get_stats(self) -> dict:
        """Get execution statistics."""
        if not self.logs:
            return {"total_calls": 0}
        total = len(self.logs)
        successes = sum(1 for r in self.logs if r.success)
        avg_time = sum(r.duration_ms for r in self.logs) / total
        by_tool = {}
        for r in self.logs:
            by_tool.setdefault(r.tool_name, 0)
            by_tool[r.tool_name] += 1
        return {
            "total_calls": total,
            "successes": successes,
            "failures": total - successes,
            "avg_duration_ms": round(avg_time, 1),
            "calls_by_tool": by_tool
        }


class RateLimitMiddleware(ToolMiddleware):
    """Prevents tool calls from exceeding rate limits."""
    
    def __init__(self, max_calls_per_minute: int = 30):
        self.max_calls = max_calls_per_minute
        self.call_times: list[float] = []
    
    def before(self, tool_name: str, arguments: dict) -> Optional[str]:
        now = time.time()
        # Remove calls older than 1 minute
        self.call_times = [t for t in self.call_times if now - t < 60]
        
        if len(self.call_times) >= self.max_calls:
            return json.dumps({
                "error": f"Rate limit exceeded: {self.max_calls} calls/minute. Try again shortly."
            })
        
        self.call_times.append(now)
        return None
    
    def after(self, tool_name: str, arguments: dict, result: str, duration_ms: float) -> str:
        return result


class OutputTrimMiddleware(ToolMiddleware):
    """Trims tool output to prevent context window overflow."""
    
    def __init__(self, max_chars: int = 4000):
        self.max_chars = max_chars
    
    def before(self, tool_name: str, arguments: dict) -> Optional[str]:
        return None
    
    def after(self, tool_name: str, arguments: dict, result: str, duration_ms: float) -> str:
        if len(result) > self.max_chars:
            truncated = result[:self.max_chars]
            return truncated + f'\n... [TRUNCATED: {len(result)} chars → {self.max_chars} chars]'
        return result


class AuthMiddleware(ToolMiddleware):
    """Controls which tools are allowed based on permission levels."""
    
    def __init__(self, allowed_tools: set[str] = None, blocked_tools: set[str] = None):
        self.allowed_tools = allowed_tools  # whitelist (if set, only these allowed)
        self.blocked_tools = blocked_tools or set()  # blacklist
    
    def before(self, tool_name: str, arguments: dict) -> Optional[str]:
        if self.allowed_tools is not None and tool_name not in self.allowed_tools:
            return json.dumps({"error": f"Tool '{tool_name}' is not authorized"})
        if tool_name in self.blocked_tools:
            return json.dumps({"error": f"Tool '{tool_name}' is blocked by policy"})
        return None
    
    def after(self, tool_name: str, arguments: dict, result: str, duration_ms: float) -> str:
        return result


# ============================================================
# The Production Tool Registry
# ============================================================

class ProductionToolRegistry:
    """Full-featured tool registry with middleware pipeline and categories."""
    
    def __init__(self):
        self.tools: dict[str, Callable] = {}
        self.schemas: dict[str, dict] = {}
        self.categories: dict[str, list[str]] = {}
        self.middleware: list[ToolMiddleware] = []
        self.pydantic_models: dict[str, type[BaseModel]] = {}
    
    def add_middleware(self, middleware: ToolMiddleware):
        """Add a middleware layer to the execution pipeline."""
        self.middleware.append(middleware)
    
    def register(self, func: Callable, category: str = "general",
                 input_model: type[BaseModel] = None):
        """Register a tool function (decorated with @tool or with .schema attribute)."""
        if not hasattr(func, 'schema'):
            raise ValueError(f"Function {func.__name__} must be decorated with @tool")
        
        name = func.tool_name
        self.tools[name] = func
        self.schemas[name] = func.schema
        self.categories.setdefault(category, []).append(name)
        
        if input_model:
            self.pydantic_models[name] = input_model
    
    def execute(self, tool_name: str, arguments: dict) -> str:
        """Execute a tool with full middleware pipeline."""
        if tool_name not in self.tools:
            return json.dumps({"error": f"Unknown tool: '{tool_name}'"})
        
        # --- Before middleware ---
        for mw in self.middleware:
            error = mw.before(tool_name, arguments)
            if error:
                return error
        
        # --- Pydantic validation ---
        if tool_name in self.pydantic_models:
            try:
                validated = self.pydantic_models[tool_name](**arguments)
                arguments = validated.model_dump()
            except Exception as e:
                return json.dumps({"error": f"Validation failed: {e}"})
        
        # --- Execute tool ---
        start = time.perf_counter()
        try:
            result = self.tools[tool_name](**arguments)
        except Exception as e:
            result = json.dumps({"error": f"Tool execution failed: {e}"})
        duration_ms = (time.perf_counter() - start) * 1000
        
        # --- After middleware ---
        for mw in self.middleware:
            result = mw.after(tool_name, arguments, result, duration_ms)
        
        return result
    
    def get_schemas(self, category: str = None) -> list[dict]:
        """Get tool schemas, optionally filtered by category."""
        if category:
            tool_names = self.categories.get(category, [])
            return [self.schemas[n] for n in tool_names if n in self.schemas]
        return list(self.schemas.values())
    
    def list_tools(self) -> dict[str, list[str]]:
        """List all registered tools by category."""
        return {cat: tools for cat, tools in self.categories.items()}


# ============================================================
# Demo: Build and use the Production Registry
# ============================================================

# Create registry
registry = ProductionToolRegistry()

# Add middleware layers (order matters!)
logger = LoggingMiddleware()
registry.add_middleware(AuthMiddleware(blocked_tools={"dangerous_tool"}))
registry.add_middleware(RateLimitMiddleware(max_calls_per_minute=60))
registry.add_middleware(logger)
registry.add_middleware(OutputTrimMiddleware(max_chars=2000))

# Register our @tool-decorated functions
registry.register(get_stock_price, category="finance")
registry.register(search_knowledge_base, category="search")
registry.register(safe_calculate, category="utility")
registry.register(create_reminder, category="productivity")
registry.register(get_user_profile, category="user_management")

print("=== Registered Tools by Category ===")
for cat, tools in registry.list_tools().items():
    print(f"  {cat}: {', '.join(tools)}")

print(f"\nTotal tools: {sum(len(t) for t in registry.list_tools().values())}")
print(f"Middleware layers: {len(registry.middleware)}")

# Execute some tools through the middleware pipeline
print("\n=== Tool Execution with Middleware ===\n")
print("--- Stock Price ---")
result = registry.execute("get_stock_price", {"symbol": "AAPL"})
print(f"  Result: {result}\n")

print("--- Knowledge Base Search ---")
result = registry.execute("search_knowledge_base", {"query": "how to get started", "max_results": 2})
print(f"  Result: {result}\n")

print("--- Calculator ---")
result = registry.execute("calculate_math", {"expression": "3.14159 * 10 * 10"})
print(f"  Result: {result}\n")

print("--- Unknown Tool ---")
result = registry.execute("nonexistent_tool", {})
print(f"  Result: {result}\n")

print("\n=== Execution Statistics ===")
stats = logger.get_stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

=== Registered Tools by Category ===
  finance: get_stock_price
  search: search_knowledge_base
  utility: calculate_math
  productivity: create_reminder
  user_management: get_user_profile

Total tools: 5
Middleware layers: 4

=== Tool Execution with Middleware ===

--- Stock Price ---
  🔧 Calling: get_stock_price({"symbol": "AAPL"}...)
  ✅ get_stock_price completed in 0.8ms
  Result: {"symbol": "AAPL", "price": 178.5, "change": "+1.2%"}

--- Knowledge Base Search ---
  🔧 Calling: search_knowledge_base({"query": "how to get started", "max_results": 2}...)
  ✅ search_knowledge_base completed in 0.1ms
  Result: {"results": [{"title": "Getting Started Guide", "snippet": "Welcome to our platform..."}, {"title": "API Reference", "snippet": "REST API endpoints for..."}], "total": 3}

--- Calculator ---
  🔧 Calling: calculate_math({"expression": "3.14159 * 10 * 10"}...)
  ✅ calculate_math completed in 0.1ms
  Result: {"expression": "3.14159 * 10 * 10", "result": 314.159}

--- Unknown Tool --

### 4.2: Tool Composition & Chaining

Sometimes tools need to call other tools, or you need to create **composite tools** that combine multiple actions. This is how complex agent workflows emerge.

In [ ]:
# ============================================================
# Section 4.2: Tool Composition & Chaining
# ============================================================

class ToolChain:
    """Chain multiple tools together — output of one feeds into the next.
    
    This is the primitive behind LangChain's "chains" and 
    agent multi-step tool use.
    """
    
    def __init__(self, registry: ProductionToolRegistry):
        self.registry = registry
        self.steps: list[dict] = []
    
    def add_step(self, tool_name: str, arg_mapper: Callable[[dict], dict]):
        """Add a step to the chain.
        
        Args:
            tool_name: Which tool to call
            arg_mapper: Function that takes previous result and returns args for this tool
        """
        self.steps.append({"tool": tool_name, "mapper": arg_mapper})
        return self  # enable chaining
    
    def execute(self, initial_input: dict) -> list[dict]:
        """Execute the chain, passing results forward."""
        results = []
        current_data = initial_input
        
        print(f"🔗 Executing chain with {len(self.steps)} steps\n")
        
        for i, step in enumerate(self.steps):
            # Map previous output to current input
            args = step["mapper"](current_data)
            
            print(f"  Step {i+1}: {step['tool']}({json.dumps(args)[:80]}...)")
            
            # Execute through registry (with middleware)
            result_str = self.registry.execute(step["tool"], args)
            result_data = json.loads(result_str)
            
            results.append({
                "step": i + 1,
                "tool": step["tool"],
                "args": args,
                "result": result_data
            })
            
            # Pass result forward
            current_data = result_data
            
            # Stop if error
            if "error" in result_data:
                print(f"  ❌ Chain stopped at step {i+1}: {result_data['error']}")
                break
        
        print(f"\n🏁 Chain completed: {len(results)} steps executed")
        return results


# --- Demo: Research Chain ---
# "Look up a user, then search knowledge base based on their plan"

chain = ToolChain(registry)
chain.add_step(
    "get_user_profile",
    lambda data: {"user_id": data["user_id"]}
)
chain.add_step(
    "search_knowledge_base",
    lambda data: {"query": f"features for {data.get('plan', 'free')} plan", "max_results": 3}
)

print("=== Research Chain: User → Knowledge Base ===\n")
results = chain.execute({"user_id": "user_001"})
print("\n--- Chain Results ---")
for r in results:
    print(f"Step {r['step']} ({r['tool']}): {json.dumps(r['result'])[:100]}")


# --- Demo: Calculation Chain ---
print("\n\n=== Calculation Chain ===\n")
calc_chain = ToolChain(registry)
calc_chain.add_step(
    "calculate_math",
    lambda data: {"expression": data["expression"]}
)
calc_chain.add_step(
    "calculate_math",
    lambda data: {"expression": f"{data['result']} * 2"}
)
calc_chain.add_step(
    "calculate_math",
    lambda data: {"expression": f"{data['result']} + 100"}
)

results = calc_chain.execute({"expression": "5 * 5"})
print("\n--- Chain Results ---")
for r in results:
    print(f"Step {r['step']}: {r['args']['expression']} = {r['result'].get('result', r['result'])}")

---
# Section 5: MCP (Model Context Protocol) — Deep Dive

## What is MCP?

**MCP** (Model Context Protocol) is an open standard created by Anthropic that defines how AI agents connect to external tools and data sources. Think of it as **USB for AI** — a universal plug that lets any agent use any tool.

```
BEFORE MCP:                              AFTER MCP:
                                         
Agent A ──custom──► Tool 1               Agent A ─┐
Agent A ──custom──► Tool 2                        │
Agent B ──custom──► Tool 1               Agent B ─┤──► MCP ──► Tool 1
Agent B ──custom──► Tool 3                        │           Tool 2
Agent C ──custom──► Tool 2               Agent C ─┘           Tool 3
                                         
= 6 custom integrations                  = 1 standard protocol
```

## MCP Architecture

```
┌────────────────┐        ┌────────────────┐        ┌────────────────┐
│   MCP HOST     │        │   MCP CLIENT   │        │   MCP SERVER   │
│                │        │                │        │                │
│  Your agent    │───────►│  Protocol      │───────►│  Exposes:      │
│  application   │◄───────│  handler       │◄───────│  • Tools       │
│                │        │                │        │  • Resources   │
│  Claude,       │        │  Manages       │        │  • Prompts     │
│  GPT, custom   │        │  connection    │        │                │
└────────────────┘        └────────────────┘        └────────────────┘
                                                            │
                          Transport Layer                    │
                          ─────────────                     ▼
                          • stdio (local processes)    External Systems
                          • HTTP+SSE (remote servers)  (APIs, DBs, files)
```

## MCP Capabilities

| Capability | What It Does | Example |
|-----------|-------------|---------|
| **Tools** | Functions the agent can call | `search_database()`, `send_email()` |
| **Resources** | Read-only data the agent can access | Files, DB records, API responses |
| **Prompts** | Reusable prompt templates | "Analyze this code", "Summarize document" |
| **Sampling** | Server can request LLM completions | Server asks agent to reason about data |

## MCP Message Format

```json
{
  "jsonrpc": "2.0",
  "method": "tools/call",
  "params": {
    "name": "search_products",
    "arguments": {
      "query": "wireless headphones",
      "max_results": 5
    }
  },
  "id": 1
}
```

MCP uses JSON-RPC 2.0 — the same protocol behind VS Code's LSP (Language Server Protocol). If you understand REST APIs, MCP is similar but bidirectional and stateful.

In [ ]:
# ============================================================
# Section 5: Understanding MCP Messages (Hands-on)
# ============================================================

# Let's implement the MCP JSON-RPC message format from scratch
# to understand exactly what flows over the wire.

@dataclass
class MCPMessage:
    """A JSON-RPC 2.0 message used by MCP."""
    jsonrpc: str = "2.0"
    method: Optional[str] = None
    params: Optional[dict] = None
    result: Optional[Any] = None
    error: Optional[dict] = None
    id: Optional[int] = None
    
    def to_json(self) -> str:
        msg = {"jsonrpc": self.jsonrpc}
        if self.method:
            msg["method"] = self.method
        if self.params is not None:
            msg["params"] = self.params
        if self.result is not None:
            msg["result"] = self.result
        if self.error is not None:
            msg["error"] = self.error
        if self.id is not None:
            msg["id"] = self.id
        return json.dumps(msg, indent=2)
    
    @classmethod
    def from_json(cls, data: str) -> "MCPMessage":
        obj = json.loads(data)
        return cls(**{k: v for k, v in obj.items() if k in cls.__dataclass_fields__})


# --- Simulate MCP message exchange ---

print("=" * 60)
print("MCP MESSAGE FLOW: Client ↔ Server")
print("=" * 60)

# Step 1: Initialize connection
print("\n--- 1. Client → Server: Initialize ---")
init_request = MCPMessage(
    method="initialize",
    params={
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "roots": {"listChanged": True}
        },
        "clientInfo": {
            "name": "my-agent",
            "version": "1.0.0"
        }
    },
    id=1
)
print(init_request.to_json())

print("\n--- 2. Server → Client: Initialize Response ---")
init_response = MCPMessage(
    result={
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "tools": {"listChanged": True},
            "resources": {"subscribe": True}
        },
        "serverInfo": {
            "name": "product-catalog-server",
            "version": "1.0.0"
        }
    },
    id=1
)
print(init_response.to_json())

# Step 2: List available tools
print("\n--- 3. Client → Server: List Tools ---")
list_tools_req = MCPMessage(method="tools/list", params={}, id=2)
print(list_tools_req.to_json())

print("\n--- 4. Server → Client: Tool List ---")
list_tools_resp = MCPMessage(
    result={
        "tools": [
            {
                "name": "search_products",
                "description": "Search the product catalog by keyword",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search terms"},
                        "max_results": {"type": "integer", "default": 10}
                    },
                    "required": ["query"]
                }
            },
            {
                "name": "get_product_details",
                "description": "Get full details for a specific product",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "product_id": {"type": "integer", "description": "Product ID"}
                    },
                    "required": ["product_id"]
                }
            }
        ]
    },
    id=2
)
print(list_tools_resp.to_json())

# Step 3: Call a tool
print("\n--- 5. Client → Server: Call Tool ---")
call_tool_req = MCPMessage(
    method="tools/call",
    params={
        "name": "search_products",
        "arguments": {"query": "headphones", "max_results": 3}
    },
    id=3
)
print(call_tool_req.to_json())

print("\n--- 6. Server → Client: Tool Result ---")
call_tool_resp = MCPMessage(
    result={
        "content": [
            {
                "type": "text",
                "text": json.dumps({
                    "products": [
                        {"id": 1, "name": "Wireless Headphones", "price": 79.99},
                        {"id": 7, "name": "Noise-Cancelling Headphones", "price": 249.99}
                    ]
                })
            }
        ]
    },
    id=3
)
print(call_tool_resp.to_json())

print("\n" + "=" * 60)
print("KEY INSIGHT: MCP is just JSON-RPC messages over a transport.")
print("The 'magic' is standardization — every server speaks the same protocol.")
print("=" * 60)

---
# Section 6: Building an MCP Server from Scratch

## What We'll Build

A complete MCP server with **6 tools** organized in 3 categories:

```
Product Catalog MCP Server
├── Search Tools
│   ├── search_products     — Search by keyword
│   └── get_product_details — Get details by ID
│
├── Analytics Tools
│   ├── get_sales_stats     — Sales statistics
│   └── get_trending        — Trending products
│
└── Action Tools
    ├── add_to_cart          — Add product to shopping cart
    └── submit_review       — Submit a product review
```

We'll first build a **simulated MCP server** that implements the full protocol in-memory (no external dependencies needed), then show how to use the official `mcp` SDK.

In [ ]:
# ============================================================
# Section 6: Building an MCP Server from Scratch
# ============================================================

# --- 6.1: The Product Database (our "external system") ---

PRODUCTS = [
    {"id": 1, "name": "Wireless Headphones", "category": "electronics", 
     "price": 79.99, "rating": 4.5, "stock": 150, "sales_30d": 423},
    {"id": 2, "name": "Python Crash Course", "category": "books", 
     "price": 29.99, "rating": 4.8, "stock": 89, "sales_30d": 567},
    {"id": 3, "name": "USB-C Hub 7-in-1", "category": "electronics", 
     "price": 34.99, "rating": 4.2, "stock": 234, "sales_30d": 312},
    {"id": 4, "name": "Mechanical Keyboard RGB", "category": "electronics", 
     "price": 129.99, "rating": 4.7, "stock": 67, "sales_30d": 189},
    {"id": 5, "name": "AI Engineering Book", "category": "books", 
     "price": 49.99, "rating": 4.9, "stock": 45, "sales_30d": 892},
    {"id": 6, "name": "Cotton T-Shirt Pack", "category": "clothing", 
     "price": 19.99, "rating": 4.0, "stock": 500, "sales_30d": 1203},
    {"id": 7, "name": "Smart Watch Pro", "category": "electronics", 
     "price": 199.99, "rating": 4.6, "stock": 78, "sales_30d": 345},
    {"id": 8, "name": "Hiking Boots Waterproof", "category": "clothing", 
     "price": 89.99, "rating": 4.3, "stock": 123, "sales_30d": 156},
    {"id": 9, "name": "Noise-Cancelling Earbuds", "category": "electronics", 
     "price": 149.99, "rating": 4.8, "stock": 92, "sales_30d": 678},
    {"id": 10, "name": "Design Patterns Book", "category": "books", 
     "price": 44.99, "rating": 4.6, "stock": 34, "sales_30d": 234},
]

# Shopping cart (in-memory state)
SHOPPING_CART: list[dict] = []

# Reviews (in-memory state)
REVIEWS: list[dict] = []


# --- 6.2: MCP Server Implementation ---

class MCPServer:
    """A complete MCP server implementation following the protocol spec.
    
    This is what the `mcp` Python SDK does under the hood.
    We build it manually so you understand every piece.
    """
    
    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self.tools: dict[str, dict] = {}  # name → {handler, schema}
        self.resources: dict[str, dict] = {}  # uri → {handler, schema}
        self._initialized = False
    
    def register_tool(self, name: str, description: str, 
                       input_schema: dict, handler: Callable):
        """Register a tool with the server."""
        self.tools[name] = {
            "name": name,
            "description": description,
            "inputSchema": input_schema,
            "handler": handler
        }
    
    def register_resource(self, uri: str, name: str, 
                           description: str, mime_type: str, handler: Callable):
        """Register a resource with the server."""
        self.resources[uri] = {
            "uri": uri,
            "name": name,
            "description": description,
            "mimeType": mime_type,
            "handler": handler
        }
    
    def handle_message(self, message: str) -> str:
        """Process an incoming JSON-RPC message and return response.
        
        This is the core message router — it dispatches to the 
        appropriate handler based on the method field.
        """
        try:
            msg = json.loads(message)
        except json.JSONDecodeError:
            return self._error_response(None, -32700, "Parse error")
        
        method = msg.get("method")
        params = msg.get("params", {})
        msg_id = msg.get("id")
        
        # Route to handler
        handlers = {
            "initialize": self._handle_initialize,
            "tools/list": self._handle_tools_list,
            "tools/call": self._handle_tools_call,
            "resources/list": self._handle_resources_list,
            "resources/read": self._handle_resources_read,
            "ping": self._handle_ping,
        }
        
        handler = handlers.get(method)
        if not handler:
            return self._error_response(msg_id, -32601, f"Method not found: {method}")
        
        try:
            result = handler(params)
            return json.dumps({
                "jsonrpc": "2.0",
                "result": result,
                "id": msg_id
            })
        except Exception as e:
            return self._error_response(msg_id, -32000, str(e))
    
    def _handle_initialize(self, params: dict) -> dict:
        self._initialized = True
        return {
            "protocolVersion": "2024-11-05",
            "capabilities": {
                "tools": {"listChanged": True},
                "resources": {"subscribe": False, "listChanged": True}
            },
            "serverInfo": {
                "name": self.name,
                "version": self.version
            }
        }
    
    def _handle_tools_list(self, params: dict) -> dict:
        return {
            "tools": [
                {
                    "name": t["name"],
                    "description": t["description"],
                    "inputSchema": t["inputSchema"]
                }
                for t in self.tools.values()
            ]
        }
    
    def _handle_tools_call(self, params: dict) -> dict:
        name = params.get("name")
        arguments = params.get("arguments", {})
        
        if name not in self.tools:
            raise ValueError(f"Unknown tool: {name}")
        
        handler = self.tools[name]["handler"]
        result = handler(**arguments)
        
        return {
            "content": [
                {"type": "text", "text": result if isinstance(result, str) else json.dumps(result)}
            ]
        }
    
    def _handle_resources_list(self, params: dict) -> dict:
        return {
            "resources": [
                {
                    "uri": r["uri"],
                    "name": r["name"],
                    "description": r["description"],
                    "mimeType": r["mimeType"]
                }
                for r in self.resources.values()
            ]
        }
    
    def _handle_resources_read(self, params: dict) -> dict:
        uri = params.get("uri")
        if uri not in self.resources:
            raise ValueError(f"Unknown resource: {uri}")
        
        handler = self.resources[uri]["handler"]
        content = handler()
        
        return {
            "contents": [
                {
                    "uri": uri,
                    "mimeType": self.resources[uri]["mimeType"],
                    "text": content if isinstance(content, str) else json.dumps(content)
                }
            ]
        }
    
    def _handle_ping(self, params: dict) -> dict:
        return {}
    
    def _error_response(self, msg_id, code: int, message: str) -> str:
        return json.dumps({
            "jsonrpc": "2.0",
            "error": {"code": code, "message": message},
            "id": msg_id
        })


# ============================================================
# 6.3: Register 6 Tools with the Server
# ============================================================

server = MCPServer("product-catalog", "1.0.0")

# --- Tool 1: Search Products ---
def mcp_search_products(query: str, category: str = None, max_results: int = 5) -> str:
    results = [p for p in PRODUCTS if query.lower() in p["name"].lower()]
    if category:
        results = [p for p in results if p["category"] == category]
    results = results[:max_results]
    return json.dumps({"query": query, "count": len(results), "products": results})

server.register_tool(
    "search_products",
    "Search the product catalog by keyword. Use when the user wants to find products.",
    {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search keywords"},
            "category": {"type": "string", "description": "Filter by category: electronics, books, clothing"},
            "max_results": {"type": "integer", "description": "Max results to return", "default": 5}
        },
        "required": ["query"]
    },
    mcp_search_products
)

# --- Tool 2: Get Product Details ---
def mcp_get_product_details(product_id: int) -> str:
    product = next((p for p in PRODUCTS if p["id"] == product_id), None)
    if not product:
        return json.dumps({"error": f"Product {product_id} not found"})
    return json.dumps(product)

server.register_tool(
    "get_product_details",
    "Get detailed information about a specific product by its ID.",
    {
        "type": "object",
        "properties": {
            "product_id": {"type": "integer", "description": "The product ID number"}
        },
        "required": ["product_id"]
    },
    mcp_get_product_details
)

# --- Tool 3: Get Sales Statistics ---
def mcp_get_sales_stats(category: str = None) -> str:
    products = PRODUCTS if not category else [p for p in PRODUCTS if p["category"] == category]
    total_sales = sum(p["sales_30d"] for p in products)
    avg_price = sum(p["price"] for p in products) / len(products) if products else 0
    top_seller = max(products, key=lambda p: p["sales_30d"]) if products else None
    return json.dumps({
        "category": category or "all",
        "total_products": len(products),
        "total_sales_30d": total_sales,
        "average_price": round(avg_price, 2),
        "top_seller": top_seller["name"] if top_seller else None,
        "top_seller_sales": top_seller["sales_30d"] if top_seller else 0
    })

server.register_tool(
    "get_sales_stats",
    "Get sales statistics and analytics. Optionally filter by product category.",
    {
        "type": "object",
        "properties": {
            "category": {"type": "string", "description": "Category to filter: electronics, books, clothing. Omit for all."}
        },
        "required": []
    },
    mcp_get_sales_stats
)

# --- Tool 4: Get Trending Products ---
def mcp_get_trending(time_period: str = "30d", limit: int = 5) -> str:
    sorted_products = sorted(PRODUCTS, key=lambda p: p["sales_30d"], reverse=True)
    trending = sorted_products[:limit]
    return json.dumps({
        "period": time_period,
        "trending": [{"rank": i+1, "name": p["name"], "sales": p["sales_30d"], "category": p["category"]} 
                     for i, p in enumerate(trending)]
    })

server.register_tool(
    "get_trending",
    "Get the top trending products by sales volume.",
    {
        "type": "object",
        "properties": {
            "time_period": {"type": "string", "description": "Time period: 7d, 30d, 90d", "default": "30d"},
            "limit": {"type": "integer", "description": "How many trending products", "default": 5}
        },
        "required": []
    },
    mcp_get_trending
)

# --- Tool 5: Add to Cart ---
def mcp_add_to_cart(product_id: int, quantity: int = 1) -> str:
    product = next((p for p in PRODUCTS if p["id"] == product_id), None)
    if not product:
        return json.dumps({"error": f"Product {product_id} not found"})
    if quantity > product["stock"]:
        return json.dumps({"error": f"Only {product['stock']} units available"})
    
    SHOPPING_CART.append({
        "product_id": product_id,
        "name": product["name"],
        "price": product["price"],
        "quantity": quantity,
        "subtotal": round(product["price"] * quantity, 2)
    })
    
    total = sum(item["subtotal"] for item in SHOPPING_CART)
    return json.dumps({
        "status": "added",
        "item": product["name"],
        "quantity": quantity,
        "cart_total": round(total, 2),
        "cart_items": len(SHOPPING_CART)
    })

server.register_tool(
    "add_to_cart",
    "Add a product to the shopping cart. This is an ACTION tool with side effects.",
    {
        "type": "object",
        "properties": {
            "product_id": {"type": "integer", "description": "Product ID to add"},
            "quantity": {"type": "integer", "description": "Quantity to add", "default": 1}
        },
        "required": ["product_id"]
    },
    mcp_add_to_cart
)

# --- Tool 6: Submit Review ---
def mcp_submit_review(product_id: int, rating: int, comment: str) -> str:
    product = next((p for p in PRODUCTS if p["id"] == product_id), None)
    if not product:
        return json.dumps({"error": f"Product {product_id} not found"})
    if not 1 <= rating <= 5:
        return json.dumps({"error": "Rating must be between 1 and 5"})
    
    review = {
        "product_id": product_id,
        "product_name": product["name"],
        "rating": rating,
        "comment": comment,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }
    REVIEWS.append(review)
    
    return json.dumps({"status": "submitted", "review": review})

server.register_tool(
    "submit_review",
    "Submit a product review with a rating (1-5) and comment. This is an ACTION tool.",
    {
        "type": "object",
        "properties": {
            "product_id": {"type": "integer", "description": "Product ID to review"},
            "rating": {"type": "integer", "description": "Rating from 1 (worst) to 5 (best)"},
            "comment": {"type": "string", "description": "Review comment text"}
        },
        "required": ["product_id", "rating", "comment"]
    },
    mcp_submit_review
)

# --- Also register a resource ---
server.register_resource(
    "catalog://categories",
    "Product Categories",
    "List of all available product categories with counts",
    "application/json",
    lambda: json.dumps({
        cat: len([p for p in PRODUCTS if p["category"] == cat])
        for cat in set(p["category"] for p in PRODUCTS)
    })
)

print(f"✅ MCP Server '{server.name}' created with {len(server.tools)} tools and {len(server.resources)} resources")
print(f"\nRegistered tools:")
for name, tool in server.tools.items():
    print(f"  🔧 {name}: {tool['description'][:60]}...")

### 6.4: Testing the MCP Server

Now let's send real MCP protocol messages to our server and see the full request/response cycle.

In [ ]:
# ============================================================
# Section 6.4: Testing MCP Server with Protocol Messages
# ============================================================

def send_to_server(server: MCPServer, method: str, params: dict = None, msg_id: int = 1) -> dict:
    """Helper to send a JSON-RPC message to the server and parse response."""
    request = json.dumps({
        "jsonrpc": "2.0",
        "method": method,
        "params": params or {},
        "id": msg_id
    })
    response = server.handle_message(request)
    return json.loads(response)


# --- Test 1: Initialize ---
print("=" * 60)
print("TEST 1: Initialize Connection")
print("=" * 60)
resp = send_to_server(server, "initialize", {
    "protocolVersion": "2024-11-05",
    "clientInfo": {"name": "test-client", "version": "1.0.0"}
})
print(f"Server: {resp['result']['serverInfo']['name']}")
print(f"Capabilities: {list(resp['result']['capabilities'].keys())}")


# --- Test 2: List Tools ---
print(f"\n{'=' * 60}")
print("TEST 2: Discover Available Tools")
print("=" * 60)
resp = send_to_server(server, "tools/list", msg_id=2)
tools = resp["result"]["tools"]
print(f"Found {len(tools)} tools:")
for t in tools:
    params = list(t["inputSchema"].get("properties", {}).keys())
    required = t["inputSchema"].get("required", [])
    print(f"  🔧 {t['name']}({', '.join(params)})")
    print(f"     Required: {required}")
    print(f"     {t['description'][:70]}")


# --- Test 3: Call tools ---
print(f"\n{'=' * 60}")
print("TEST 3: Call Tools via MCP Protocol")
print("=" * 60)

# Search
print("\n--- search_products ---")
resp = send_to_server(server, "tools/call", {
    "name": "search_products",
    "arguments": {"query": "book", "max_results": 3}
}, msg_id=3)
result = json.loads(resp["result"]["content"][0]["text"])
print(f"Found {result['count']} products matching 'book':")
for p in result["products"]:
    print(f"  #{p['id']} {p['name']} — ${p['price']} (⭐{p['rating']})")

# Analytics
print("\n--- get_sales_stats ---")
resp = send_to_server(server, "tools/call", {
    "name": "get_sales_stats",
    "arguments": {"category": "electronics"}
}, msg_id=4)
stats = json.loads(resp["result"]["content"][0]["text"])
print(f"Electronics: {stats['total_products']} products, "
      f"{stats['total_sales_30d']} sales in 30d, "
      f"avg price ${stats['average_price']}")
print(f"Top seller: {stats['top_seller']} ({stats['top_seller_sales']} sales)")

# Trending
print("\n--- get_trending ---")
resp = send_to_server(server, "tools/call", {
    "name": "get_trending",
    "arguments": {"limit": 5}
}, msg_id=5)
trending = json.loads(resp["result"]["content"][0]["text"])
print("Top 5 trending products:")
for t in trending["trending"]:
    print(f"  #{t['rank']} {t['name']} ({t['category']}) — {t['sales']} sales")

# Add to cart (side effect)
print("\n--- add_to_cart ---")
resp = send_to_server(server, "tools/call", {
    "name": "add_to_cart",
    "arguments": {"product_id": 5, "quantity": 2}
}, msg_id=6)
cart = json.loads(resp["result"]["content"][0]["text"])
print(f"Added: {cart['item']} x{cart['quantity']}, Cart total: ${cart['cart_total']}")

# Submit review (side effect)
print("\n--- submit_review ---")
resp = send_to_server(server, "tools/call", {
    "name": "submit_review",
    "arguments": {"product_id": 5, "rating": 5, "comment": "Best AI book ever!"}
}, msg_id=7)
review = json.loads(resp["result"]["content"][0]["text"])
print(f"Review submitted: ⭐{review['review']['rating']} for {review['review']['product_name']}")


# --- Test 4: Read resource ---
print(f"\n{'=' * 60}")
print("TEST 4: Read Resource")
print("=" * 60)
resp = send_to_server(server, "resources/read", {
    "uri": "catalog://categories"
}, msg_id=8)
categories = json.loads(resp["result"]["contents"][0]["text"])
print(f"Product categories: {categories}")


# --- Test 5: Error handling ---
print(f"\n{'=' * 60}")
print("TEST 5: Error Handling")
print("=" * 60)
resp = send_to_server(server, "tools/call", {
    "name": "nonexistent_tool",
    "arguments": {}
}, msg_id=9)
print(f"Unknown tool error: {resp.get('error', {}).get('message', 'N/A')}")

resp = send_to_server(server, "unknown_method", {}, msg_id=10)
print(f"Unknown method error: {resp.get('error', {}).get('message', 'N/A')}")

---
# Section 7: Building an MCP Client

## The Client's Job

The MCP client connects to an MCP server and translates between the agent's needs and the MCP protocol:

```
Agent wants to call "search_products"
        │
        ▼
MCP Client:
  1. Was tools/list called yet? → Cache schemas
  2. Build JSON-RPC request
  3. Send to server (stdio or HTTP)
  4. Parse response
  5. Return result to agent
        │
        ▼
Agent gets the tool result
```

The client also handles:
- Connection lifecycle (initialize → use → disconnect)
- Tool discovery and schema caching
- Converting MCP tool schemas to OpenAI format (so the LLM knows about them)

In [ ]:
# ============================================================
# Section 7: MCP Client Implementation
# ============================================================

class MCPClient:
    """MCP client that connects to an MCP server.
    
    In production, this would communicate over stdio or HTTP+SSE.
    Here we connect directly to our in-memory server for learning.
    """
    
    def __init__(self, client_name: str = "agent-client", version: str = "1.0.0"):
        self.client_name = client_name
        self.version = version
        self.server: Optional[MCPServer] = None
        self.server_info: Optional[dict] = None
        self.available_tools: list[dict] = []
        self.available_resources: list[dict] = []
        self._msg_id = 0
        self._connected = False
    
    def _next_id(self) -> int:
        self._msg_id += 1
        return self._msg_id
    
    def _send(self, method: str, params: dict = None) -> dict:
        """Send a JSON-RPC message and parse the response."""
        if not self.server:
            raise ConnectionError("Not connected to any server")
        
        request = json.dumps({
            "jsonrpc": "2.0",
            "method": method,
            "params": params or {},
            "id": self._next_id()
        })
        
        response_str = self.server.handle_message(request)
        response = json.loads(response_str)
        
        if "error" in response:
            raise RuntimeError(f"MCP Error [{response['error']['code']}]: {response['error']['message']}")
        
        return response.get("result", {})
    
    def connect(self, server: MCPServer):
        """Connect to an MCP server and initialize the session."""
        self.server = server
        
        # Initialize
        result = self._send("initialize", {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {
                "name": self.client_name,
                "version": self.version
            }
        })
        
        self.server_info = result.get("serverInfo", {})
        self._connected = True
        
        # Discover tools
        tools_result = self._send("tools/list")
        self.available_tools = tools_result.get("tools", [])
        
        # Discover resources
        try:
            res_result = self._send("resources/list")
            self.available_resources = res_result.get("resources", [])
        except RuntimeError:
            self.available_resources = []
        
        print(f"✅ Connected to '{self.server_info.get('name', '?')}'")
        print(f"   Version: {self.server_info.get('version', '?')}")
        print(f"   Tools: {len(self.available_tools)}")
        print(f"   Resources: {len(self.available_resources)}")
    
    def list_tools(self) -> list[dict]:
        """Get all available tools."""
        return self.available_tools
    
    def call_tool(self, name: str, arguments: dict = None) -> Any:
        """Call a tool by name and return the parsed result."""
        result = self._send("tools/call", {
            "name": name,
            "arguments": arguments or {}
        })
        
        # Parse the content
        content = result.get("content", [])
        if content and content[0].get("type") == "text":
            text = content[0]["text"]
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                return text
        return content
    
    def read_resource(self, uri: str) -> Any:
        """Read a resource by URI."""
        result = self._send("resources/read", {"uri": uri})
        contents = result.get("contents", [])
        if contents:
            text = contents[0].get("text", "")
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                return text
        return contents
    
    def get_openai_tools(self) -> list[dict]:
        """Convert MCP tool schemas to OpenAI function calling format.
        
        This is the KEY bridge — MCP uses its own schema format,
        but OpenAI/Anthropic APIs expect their specific format.
        """
        openai_tools = []
        for tool in self.available_tools:
            openai_tools.append({
                "type": "function",
                "function": {
                    "name": tool["name"],
                    "description": tool["description"],
                    "parameters": tool["inputSchema"]
                }
            })
        return openai_tools
    
    def disconnect(self):
        """Disconnect from the server."""
        self.server = None
        self._connected = False
        self.available_tools = []
        self.available_resources = []
        print("Disconnected from server")


# ============================================================
# Demo: Use the MCP Client
# ============================================================

client = MCPClient("my-learning-agent")
client.connect(server)

print("\n=== Available Tools ===")
for t in client.list_tools():
    print(f"  🔧 {t['name']}: {t['description'][:60]}...")

print("\n=== Call Tools via Client ===")

# Search
print("\n--- Search for electronics ---")
result = client.call_tool("search_products", {"query": "smart", "category": "electronics"})
print(f"  Found {result['count']} products")
for p in result["products"]:
    print(f"    {p['name']} — ${p['price']}")

# Analytics
print("\n--- Sales stats ---")
stats = client.call_tool("get_sales_stats")
print(f"  Total sales (30d): {stats['total_sales_30d']}")
print(f"  Top seller: {stats['top_seller']}")

# Cart
print("\n--- Add to cart ---")
cart = client.call_tool("add_to_cart", {"product_id": 9, "quantity": 1})
print(f"  {cart['status']}: {cart['item']}, total: ${cart['cart_total']}")

# Resource
print("\n--- Read resource ---")
categories = client.read_resource("catalog://categories")
print(f"  Categories: {categories}")

# Show OpenAI-compatible schemas
print("\n=== OpenAI-Compatible Tool Schemas ===")
openai_tools = client.get_openai_tools()
print(f"Generated {len(openai_tools)} OpenAI tool schemas")
print(f"\nFirst tool schema:")
print(json.dumps(openai_tools[0], indent=2)[:300] + "...")

### 7.2: Real MCP Server with the `mcp` Python SDK

Here's how you'd build the same server using Anthropic's official MCP SDK. This creates a **real** MCP server that can be used with Claude Desktop, VS Code Copilot, or any MCP-compatible client.

> **Note**: This cell shows the code structure — run it as a standalone `.py` file (not in a notebook) since it starts a server process.

In [ ]:
# ============================================================
# Section 7.2: Real MCP SDK — Reference Implementation
# ============================================================
# This shows the EXACT code you'd write using the official `mcp` SDK.
# Compare it to our from-scratch version to see how similar they are!
#
# DO NOT RUN this cell in the notebook — it starts a server.
# Instead, save this as product_catalog_server.py and run:
#   python product_catalog_server.py
#
# Then configure your MCP client (Claude Desktop, VS Code, etc.)
# to connect to it.

MCP_SERVER_CODE = '''
#!/usr/bin/env python3
"""
Product Catalog MCP Server — Built with the official MCP SDK.

Run: python product_catalog_server.py
Configure in Claude Desktop's claude_desktop_config.json:
{
  "mcpServers": {
    "product-catalog": {
      "command": "python",
      "args": ["product_catalog_server.py"]
    }
  }
}
"""
from mcp.server.fastmcp import FastMCP
import json

# Create the server
mcp = FastMCP("product-catalog")

# Product database
PRODUCTS = [
    {"id": 1, "name": "Wireless Headphones", "category": "electronics", "price": 79.99, "rating": 4.5},
    {"id": 2, "name": "Python Crash Course", "category": "books", "price": 29.99, "rating": 4.8},
    {"id": 3, "name": "USB-C Hub 7-in-1", "category": "electronics", "price": 34.99, "rating": 4.2},
    {"id": 4, "name": "AI Engineering Book", "category": "books", "price": 49.99, "rating": 4.9},
    {"id": 5, "name": "Smart Watch Pro", "category": "electronics", "price": 199.99, "rating": 4.6},
]

CART = []


@mcp.tool()
def search_products(query: str, category: str = None, max_results: int = 5) -> str:
    """Search the product catalog by keyword.
    
    Args:
        query: Search keywords to match product names
        category: Optional filter: electronics, books, clothing
        max_results: Maximum number of results to return
    """
    results = [p for p in PRODUCTS if query.lower() in p["name"].lower()]
    if category:
        results = [p for p in results if p["category"] == category]
    return json.dumps({"count": len(results[:max_results]), "products": results[:max_results]})


@mcp.tool()
def get_product_details(product_id: int) -> str:
    """Get detailed information about a specific product.
    
    Args:
        product_id: The product ID number
    """
    product = next((p for p in PRODUCTS if p["id"] == product_id), None)
    if not product:
        return json.dumps({"error": f"Product {product_id} not found"})
    return json.dumps(product)


@mcp.tool()
def get_trending(limit: int = 5) -> str:
    """Get the top trending products by rating.
    
    Args:
        limit: How many products to return
    """
    sorted_products = sorted(PRODUCTS, key=lambda p: p["rating"], reverse=True)
    return json.dumps({"trending": sorted_products[:limit]})


@mcp.tool()
def add_to_cart(product_id: int, quantity: int = 1) -> str:
    """Add a product to the shopping cart.
    
    Args:
        product_id: Product ID to add
        quantity: How many to add
    """
    product = next((p for p in PRODUCTS if p["id"] == product_id), None)
    if not product:
        return json.dumps({"error": f"Product {product_id} not found"})
    CART.append({"product": product["name"], "quantity": quantity, "price": product["price"]})
    total = sum(item["price"] * item["quantity"] for item in CART)
    return json.dumps({"status": "added", "cart_total": round(total, 2), "items": len(CART)})


@mcp.tool()
def get_sales_stats(category: str = None) -> str:
    """Get sales analytics for the catalog.
    
    Args:
        category: Optional category filter
    """
    products = PRODUCTS if not category else [p for p in PRODUCTS if p["category"] == category]
    avg_price = sum(p["price"] for p in products) / len(products) if products else 0
    return json.dumps({
        "total_products": len(products),
        "average_price": round(avg_price, 2),
        "top_rated": max(products, key=lambda p: p["rating"])["name"] if products else None
    })


@mcp.tool()
def submit_review(product_id: int, rating: int, comment: str) -> str:
    """Submit a review for a product.
    
    Args:
        product_id: Product to review
        rating: Rating from 1 to 5
        comment: Review text
    """
    if not 1 <= rating <= 5:
        return json.dumps({"error": "Rating must be 1-5"})
    product = next((p for p in PRODUCTS if p["id"] == product_id), None)
    if not product:
        return json.dumps({"error": "Product not found"})
    return json.dumps({"status": "submitted", "product": product["name"], "rating": rating})


@mcp.resource("catalog://categories")
def get_categories() -> str:
    """List all product categories."""
    categories = list(set(p["category"] for p in PRODUCTS))
    return json.dumps(categories)


if __name__ == "__main__":
    mcp.run()
'''

print("=" * 60)
print("COMPARISON: From Scratch vs Official MCP SDK")
print("=" * 60)
print()
print("From Scratch (what we built):")
print("  server = MCPServer('product-catalog')")
print("  server.register_tool('search_products', desc, schema, handler)")
print("  response = server.handle_message(json_rpc_request)")
print()
print("Official MCP SDK:")
print("  mcp = FastMCP('product-catalog')")
print("  @mcp.tool()")
print("  def search_products(query: str) -> str: ...")
print("  mcp.run()  # starts stdio transport")
print()
print("The SDK adds:")
print("  ✅ Transport handling (stdio, SSE)")
print("  ✅ Auto schema generation from type hints")
print("  ✅ Connection lifecycle management")
print("  ✅ Error handling and protocol compliance")
print("  ✅ @mcp.tool() decorator (similar to our @tool)")
print()
print(f"Server code: {len(MCP_SERVER_CODE)} characters")
print("To use: Save as .py file and run with `python product_catalog_server.py`")

---
# Section 8: Capstone — MCP-Powered Agent

## Bringing It All Together

Now we connect everything:
1. **MCP Server** → exposes tools via the protocol
2. **MCP Client** → discovers and invokes tools
3. **Simulated LLM** → decides which tools to call (no API key needed)
4. **Agent Loop** → ReAct-style reasoning over MCP tools
5. **Middleware** → logging, rate limiting on every tool call

```
┌──────────────────────────────────────────────────────────────────┐
│                     MCP-POWERED AGENT                           │
│                                                                  │
│  User Query                                                      │
│       │                                                          │
│       ▼                                                          │
│  ┌──────────┐    tools/list     ┌──────────────┐                │
│  │  Agent    │─────────────────►│  MCP Client   │                │
│  │  Loop     │    tools/call    │               │                │
│  │  (ReAct)  │◄─────────────────│  (Protocol)   │                │
│  └──────────┘                   └───────┬───────┘                │
│       │                                 │                        │
│       │ LLM decides                     │ MCP Protocol           │
│       │ which tool                      │                        │
│       ▼                                 ▼                        │
│  ┌──────────┐                   ┌──────────────┐                │
│  │ Simulated│                   │  MCP Server   │                │
│  │ LLM      │                   │  (6 tools)    │                │
│  │ (or real)│                   │               │                │
│  └──────────┘                   └──────────────┘                │
│                                                                  │
│  Middleware Pipeline: Auth → Rate Limit → Log → Trim Output      │
└──────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Section 8: Capstone — MCP-Powered ReAct Agent
# ============================================================

@dataclass
class LLMResponse:
    """Structured LLM response with optional tool calls."""
    content: Optional[str] = None
    tool_calls: list[dict] = field(default_factory=list)
    finish_reason: str = "stop"


class MCPSimulatedLLM:
    """Simulated LLM that uses scripted plans to demonstrate the agent loop.
    
    In production, replace with: OpenAI, Anthropic, or any LLM API.
    The agent logic stays the same — only this class changes.
    """
    
    def __init__(self):
        self.plans: dict[str, list[dict]] = {}
        self._call_count: dict[str, int] = {}
    
    def add_plan(self, query_contains: str, steps: list[dict]):
        """Add a scripted plan for queries containing the given text."""
        self.plans[query_contains.lower()] = steps
        self._call_count[query_contains.lower()] = 0
    
    def chat(self, messages: list[dict], tools: list[dict] = None) -> LLMResponse:
        """Simulate an LLM chat completion with tool calling."""
        # Find the user's query
        user_msg = ""
        for m in messages:
            if m["role"] == "user":
                user_msg = m["content"]
        
        # Find matching plan
        for trigger, steps in self.plans.items():
            if trigger in user_msg.lower():
                idx = self._call_count.get(trigger, 0)
                
                if idx < len(steps):
                    self._call_count[trigger] = idx + 1
                    step = steps[idx]
                    
                    if "tool_calls" in step:
                        return LLMResponse(
                            content=step.get("thought"),
                            tool_calls=step["tool_calls"],
                            finish_reason="tool_calls"
                        )
                    else:
                        return LLMResponse(
                            content=step["content"],
                            finish_reason="stop"
                        )
        
        return LLMResponse(content="I don't have a plan for that query.", finish_reason="stop")


class MCPAgent:
    """A ReAct agent that uses MCP for tool discovery and execution.
    
    This is the COMPLETE agent:
    - Discovers tools via MCP client
    - Sends tool schemas to LLM
    - Executes tool calls through MCP
    - Logs everything via middleware
    - Runs the full ReAct loop
    """
    
    def __init__(self, llm: MCPSimulatedLLM, mcp_client: MCPClient,
                 max_iterations: int = 10, verbose: bool = True):
        self.llm = llm
        self.mcp_client = mcp_client
        self.max_iterations = max_iterations
        self.verbose = verbose
        self.trace: list[dict] = []
    
    def run(self, query: str) -> str:
        """Execute the agent loop for a user query."""
        self.trace = []
        
        # Get available tools from MCP (auto-discovered!)
        openai_tools = self.mcp_client.get_openai_tools()
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"🤖 MCP Agent — Processing: {query}")
            print(f"{'='*60}")
            print(f"📦 Discovered {len(openai_tools)} tools via MCP")
        
        # Build initial messages
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful shopping assistant with access to a product catalog. "
                    "Use the available tools to help users find products, check trends, "
                    "manage their cart, and get analytics. Always explain your reasoning."
                )
            },
            {"role": "user", "content": query}
        ]
        
        # ReAct Loop
        for iteration in range(1, self.max_iterations + 1):
            if self.verbose:
                print(f"\n--- Iteration {iteration} ---")
            
            # REASON: Ask LLM what to do
            response = self.llm.chat(messages, tools=openai_tools)
            
            # If LLM has reasoning text, show it
            if response.content and response.tool_calls:
                if self.verbose:
                    print(f"💭 Thought: {response.content}")
                self.trace.append({"type": "thought", "content": response.content})
            
            # STOP: No tool calls → final answer
            if response.finish_reason == "stop" or not response.tool_calls:
                if self.verbose:
                    print(f"\n✅ Final Answer: {response.content}")
                self.trace.append({"type": "answer", "content": response.content})
                return response.content
            
            # ACT: Execute tool calls through MCP
            for tc in response.tool_calls:
                tool_name = tc["name"]
                tool_args = tc.get("arguments", {})
                
                if self.verbose:
                    print(f"🔧 Action: {tool_name}({json.dumps(tool_args)[:80]})")
                
                # Execute through MCP client
                try:
                    result = self.mcp_client.call_tool(tool_name, tool_args)
                    result_str = json.dumps(result) if isinstance(result, dict) else str(result)
                except Exception as e:
                    result_str = json.dumps({"error": str(e)})
                
                if self.verbose:
                    display = result_str[:150] + "..." if len(result_str) > 150 else result_str
                    print(f"👁️  Observation: {display}")
                
                self.trace.append({
                    "type": "tool_call",
                    "tool": tool_name,
                    "args": tool_args,
                    "result": result_str
                })
                
                # Add tool result to messages for next LLM call
                messages.append({
                    "role": "assistant",
                    "content": None,
                    "tool_calls": [{"id": f"call_{iteration}", "type": "function",
                                   "function": {"name": tool_name, "arguments": json.dumps(tool_args)}}]
                })
                messages.append({
                    "role": "tool",
                    "tool_call_id": f"call_{iteration}",
                    "content": result_str
                })
        
        return "Max iterations reached without a final answer."
    
    def get_trace_summary(self) -> str:
        """Get a formatted summary of the agent's execution trace."""
        lines = ["Agent Trace:"]
        for i, step in enumerate(self.trace):
            if step["type"] == "thought":
                lines.append(f"  {i+1}. 💭 Thought: {step['content']}")
            elif step["type"] == "tool_call":
                lines.append(f"  {i+1}. 🔧 Called: {step['tool']}({json.dumps(step['args'])[:50]})")
            elif step["type"] == "answer":
                lines.append(f"  {i+1}. ✅ Answer: {step['content'][:80]}...")
        return "\n".join(lines)


# ============================================================
# Create the MCP-Powered Agent
# ============================================================

# LLM with pre-scripted plans (replace with real LLM for production)
llm = MCPSimulatedLLM()

# Plan 1: "what's trending" → get_trending → format answer
llm.add_plan("trending", [
    {
        "thought": "The user wants to know about trending products. Let me check the trending data.",
        "tool_calls": [{"name": "get_trending", "arguments": {"limit": 5}}]
    },
    {
        "content": "Here are the top 5 trending products:\n"
                   "1. 🥇 Cotton T-Shirt Pack — 1,203 sales (clothing)\n"
                   "2. 🥈 AI Engineering Book — 892 sales (books)\n"
                   "3. 🥉 Noise-Cancelling Earbuds — 678 sales (electronics)\n"
                   "4. Python Crash Course — 567 sales (books)\n"
                   "5. Wireless Headphones — 423 sales (electronics)\n\n"
                   "Books and electronics are dominating the trends!"
    }
])

# Plan 2: "find me headphones" → search → get details → answer
llm.add_plan("headphones", [
    {
        "thought": "User wants headphones. Let me search the catalog.",
        "tool_calls": [{"name": "search_products", "arguments": {"query": "headphones", "category": "electronics"}}]
    },
    {
        "thought": "Found some options. Let me get details on the top-rated one.",
        "tool_calls": [{"name": "get_product_details", "arguments": {"product_id": 9}}]
    },
    {
        "content": "I found 2 headphone options for you:\n\n"
                   "1. **Noise-Cancelling Earbuds** — $149.99 (⭐4.8)\n"
                   "   Best choice for quality, excellent noise cancellation\n\n"
                   "2. **Wireless Headphones** — $79.99 (⭐4.5)\n"
                   "   Great budget option with solid reviews\n\n"
                   "Would you like me to add either to your cart?"
    }
])

# Plan 3: "add the earbuds to cart and show stats" → add_to_cart → get_sales_stats → answer
llm.add_plan("add.*cart", [
    {
        "thought": "User wants to add earbuds to cart. Let me do that.",
        "tool_calls": [{"name": "add_to_cart", "arguments": {"product_id": 9, "quantity": 1}}]
    },
    {
        "thought": "Added to cart. Now let me get the sales stats they asked for.",
        "tool_calls": [{"name": "get_sales_stats", "arguments": {}}]
    },
    {
        "content": "Done! Here's your update:\n\n"
                   "🛒 **Cart Updated**: Noise-Cancelling Earbuds added ($149.99)\n\n"
                   "📊 **Store Stats**:\n"
                   "- 10 products in catalog\n"
                   "- Average price: $87.89\n"
                   "- Top seller: AI Engineering Book (892 sales in 30 days)\n"
                   "- Total catalog sales: 4,999 units this month"
    }
])

# Create agent with MCP client
agent = MCPAgent(llm=llm, mcp_client=client, verbose=True)

print("✅ MCP-Powered Agent ready!")
print(f"   LLM: Simulated (3 scripted scenarios)")
print(f"   MCP Tools: {len(client.available_tools)} available")

### 8.2: Run the Agent — Scenario Tests

Let's run all three scenarios and watch the MCP-powered agent in action.

In [ ]:
# ============================================================
# Section 8.2: Run the MCP Agent — All Scenarios
# ============================================================

# Reset cart for clean demo
SHOPPING_CART.clear()

# --- Scenario 1: Discovery query ---
print("\n" + "🔵" * 30)
print("SCENARIO 1: Product Discovery")
print("🔵" * 30)
result1 = agent.run("What's trending right now?")

# Reset LLM call counts for next scenario
llm._call_count = {k: 0 for k in llm._call_count}

# --- Scenario 2: Multi-step research ---
print("\n\n" + "🟢" * 30)
print("SCENARIO 2: Multi-Step Research")
print("🟢" * 30)
result2 = agent.run("Find me some headphones")

# Reset
llm._call_count = {k: 0 for k in llm._call_count}

# --- Scenario 3: Action + Analytics ---
print("\n\n" + "🟠" * 30)
print("SCENARIO 3: Action + Analytics") 
print("🟠" * 30)
result3 = agent.run("Add the earbuds to my cart and show me store stats")

# --- Summary ---
print("\n\n" + "=" * 60)
print("EXECUTION SUMMARY")
print("=" * 60)
print(f"\nScenario 1 (Trending): {len([t for t in agent.trace if t['type'] == 'tool_call'])} tool calls")

# Show full trace for the last scenario
print(f"\n{agent.get_trace_summary()}")

### 8.3: Connect to a Real LLM (OpenAI)

Replace the simulated LLM with a real one — the agent code stays identical. Only the LLM class changes.

In [ ]:
# ============================================================
# Section 8.3: Real LLM Integration
# ============================================================
# Uncomment and run this cell when you have an OpenAI API key.
# The ONLY change is swapping MCPSimulatedLLM → OpenAILLM.
# Everything else (MCP server, client, agent loop) is identical!

"""
import os
from openai import OpenAI

class OpenAILLM:
    '''Real LLM using OpenAI's API — drop-in replacement for MCPSimulatedLLM.'''
    
    def __init__(self, model: str = "gpt-4o-mini"):
        self.client = OpenAI()  # uses OPENAI_API_KEY env var
        self.model = model
    
    def chat(self, messages: list[dict], tools: list[dict] = None) -> LLMResponse:
        kwargs = {"model": self.model, "messages": messages}
        if tools:
            kwargs["tools"] = tools
            kwargs["tool_choice"] = "auto"
        
        response = self.client.chat.completions.create(**kwargs)
        msg = response.choices[0].message
        
        tool_calls = []
        if msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls.append({
                    "name": tc.function.name,
                    "arguments": json.loads(tc.function.arguments)
                })
        
        return LLMResponse(
            content=msg.content,
            tool_calls=tool_calls,
            finish_reason=response.choices[0].finish_reason
        )


# Create real agent with same MCP client
real_llm = OpenAILLM("gpt-4o-mini")
real_agent = MCPAgent(llm=real_llm, mcp_client=client, verbose=True)

# Now the agent works with real LLM reasoning!
result = real_agent.run("What are the best-rated electronics under $100?")
"""

print("=" * 60)
print("Real LLM Integration (commented out — needs API key)")
print("=" * 60)
print()
print("To use with a real LLM:")
print("  1. pip install openai")
print("  2. export OPENAI_API_KEY='sk-...'")
print("  3. Uncomment the code above and run")
print()
print("What changes:  MCPSimulatedLLM → OpenAILLM")
print("What stays:    MCP Server, MCP Client, Agent Loop, Middleware")
print()
print("This is the power of clean architecture:")
print("  → Swap the brain, keep the body.")
print("  → Swap the tools, keep the brain.")

### 8.4: Architecture Diagram — What You Built vs What Frameworks Do

Let's see the full picture of how today's work maps to production agent systems.

In [ ]:
# ============================================================
# Section 8.4: Complete Architecture Map
# ============================================================

architecture = """
╔══════════════════════════════════════════════════════════════════╗
║              WEEK 4 — COMPLETE ARCHITECTURE MAP                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  YOUR CODE                    FRAMEWORK EQUIVALENT               ║
║  ─────────                    ────────────────────               ║
║                                                                  ║
║  @tool decorator         →    LangChain @tool                    ║
║  parse_docstring_params  →    LangChain create_tool_from_func    ║
║  pydantic_to_openai_schema →  OpenAI's function schema gen       ║
║                                                                  ║
║  ToolMiddleware (ABC)    →    LangChain callbacks                ║
║  LoggingMiddleware       →    LangSmith / Langfuse               ║
║  RateLimitMiddleware     →    API gateway rate limiting          ║
║  AuthMiddleware          →    OAuth / RBAC layers                ║
║  OutputTrimMiddleware    →    Framework output parsers           ║
║                                                                  ║
║  ProductionToolRegistry  →    LangChain ToolNode / CrewAI Tool   ║
║  ToolChain               →    LangChain RunnableSequence         ║
║                                                                  ║
║  MCPServer               →    mcp.server.FastMCP                 ║
║  MCPClient               →    mcp.client.ClientSession           ║
║  handle_message()        →    MCP transport handler              ║
║                                                                  ║
║  MCPAgent (ReAct loop)   →    LangGraph Agent + MCP adapter      ║
║  MCPSimulatedLLM         →    Testing mock / FakeLLM             ║
║  OpenAILLM (real)        →    ChatOpenAI wrapper                 ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  DATA FLOW:                                                      ║
║                                                                  ║
║  User Query → Agent Loop → LLM (reason) → MCP Client           ║
║                    ↑                          │                   ║
║                    │                    tools/call                ║
║                    │                          │                   ║
║                    │                    MCP Server                ║
║                    │                          │                   ║
║                    │                    Tool Function              ║
║                    │                          │                   ║
║                    │                    Middleware                 ║
║                    │                    (log, auth,               ║
║                    │                     rate limit)              ║
║                    │                          │                   ║
║                    └──── observation ─────────┘                   ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""

print(architecture)

# Quick stats on what we built
print("📊 Week 4 By The Numbers:")
print(f"   Classes built: 12")
print(f"   @tool functions: 5") 
print(f"   MCP Server tools: 6")
print(f"   Middleware layers: 4 (Auth, RateLimit, Logging, OutputTrim)")
print(f"   MCP protocol methods handled: 6")
print(f"   Agent scenarios tested: 3")
print(f"   Lines of production-quality code: ~800")

---
# ✅ Week 4 Complete!

## What You Built from Scratch

| Component | What You Implemented | Framework Equivalent |
|-----------|---------------------|----------------------|
| **@tool Decorator** | Auto-generates OpenAI schemas from type hints + docstrings | LangChain `@tool`, CrewAI `Tool` |
| **Pydantic Tools** | Type-safe tools with input validation, injection prevention | OpenAI structured outputs |
| **Pydantic → Schema** | Convert Pydantic models to OpenAI function calling format | LangChain `create_tool_from_func` |
| **Tool Middleware** | Pluggable layers: logging, auth, rate limit, output trim | LangChain callbacks, middleware |
| **Tool Registry** | Production registry with categories, middleware pipeline | LangChain `ToolNode`, CrewAI tools |
| **Tool Chaining** | Sequential tool composition with result forwarding | LangChain `RunnableSequence` |
| **MCP Server** | Full JSON-RPC 2.0 server with 6 tools + resources | `mcp.server.FastMCP` |
| **MCP Client** | Connect, discover, invoke, schema translation | `mcp.client.ClientSession` |
| **MCP Agent** | ReAct agent powered by MCP tool discovery | LangGraph + MCP adapter |

## Key Insights

```
1. Tools are the bridge between LLMs and the real world:
   LLM (reasoning) → Tool (action) → Result (observation)

2. The LLM only sees the schema, never the code:
   → Good descriptions = good tool selection
   → Bad descriptions = wrong tool calls

3. MCP standardizes what was previously custom:
   Before: Every agent × every tool = N×M integrations
   After:  Every agent → MCP → every tool = N+M integrations

4. Middleware is production-essential:
   → Logging: Debug agent failures
   → Rate limits: Don't blow API quotas
   → Auth: Control access to dangerous tools
   → Output trim: Protect context windows

5. The agent loop is TOOL-AGNOSTIC:
   → Same loop works with MCP, custom tools, or any tool protocol
   → Swap LLM, swap tools — the architecture stays clean
```

## Next: Phase 2 Continues

**Week 5: Memory Systems** — Add persistent memory to agents
- Short-term memory (conversation buffer)
- Long-term memory (vector databases: ChromaDB, FAISS)
- Episodic memory (learning from past tasks)
- Context window optimization strategies

---
*Part of the [AI Agents from Scratch Guide](AI-Agents-From-Scratch-Guide.md)*